# Semana 13 · ¿Por qué algunos planetas parecen ir hacia atrás?

Este cuaderno acompaña a la sesión de proyecto de la semana 13. Vas a mover la Tierra y Marte en órbitas circulares y a mirar Marte desde la Tierra, para ver aparecer el movimiento retrógrado.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. El seno y el coseno como coordenadas

Un punto de la circunferencia de radio 1 con ángulo $\theta$ tiene coordenadas $(\cos\theta, \sin\theta)$. La celda dibuja la circunferencia y marca varios ángulos, con su seno y su coseno. Fíjate en los signos de cada cuadrante.

Recuerda: NumPy trabaja en **radianes**. Para pasar de grados a radianes se usa `np.radians` (o se multiplica por $\pi/180$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

angulos = [30, 135, 200, 300]                 # en grados
t = np.linspace(0, 2 * np.pi, 400)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(np.cos(t), np.sin(t), color="gray")
ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
for a in angulos:
    x, y = np.cos(np.radians(a)), np.sin(np.radians(a))
    ax.plot([0, x], [0, y], lw=1)
    ax.plot(x, y, "o")
    ax.annotate(f"{a}°\ncos={x:.2f}\nsen={y:.2f}", (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_aspect("equal"); ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_title("La circunferencia unidad")
plt.show()

**Ejercicio.** Comprueba con Python que $\cos^2\theta + \sin^2\theta = 1$ para cinco ángulos cualesquiera, incluidos alguno negativo y alguno mayor que 360°.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
for a in [-40, 75, 210, 400, 1000]:
    r = np.radians(a)
    print(a, "°  →", round(np.cos(r)**2 + np.sin(r)**2, 12))

## 2. Dos planetas en órbitas circulares

Usamos el **Sol como origen** y medimos las distancias en **unidades astronómicas** (1 ua es la distancia media Tierra-Sol, unos 149,6 millones de km) y el tiempo en días. Cada planeta tiene un radio $r$, un periodo $T$ y un ángulo inicial $\theta_0$:

$$x(t) = r\cos(\theta_0 + \omega t),\qquad y(t) = r\sin(\theta_0 + \omega t),\qquad \omega = 2\pi/T.$$

Empezamos con los dos planetas alineados con el Sol (los dos con $\theta_0 = 0$): es una oposición de Marte.

In [ ]:
def posicion(r, T, theta0, t):
    """Posición (x, y) de un planeta en órbita circular. t en días, ángulos en radianes."""
    omega = 2 * np.pi / T
    ang = theta0 + omega * t
    return r * np.cos(ang), r * np.sin(ang)

r_T, T_T = 1.000, 365.256      # Tierra: radio en ua, periodo sidéreo en días
r_M, T_M = 1.524, 686.980      # Marte

dias = np.arange(0, 3 * 365)   # tres años, un punto por día
xT, yT = posicion(r_T, T_T, 0.0, dias)
xM, yM = posicion(r_M, T_M, 0.0, dias)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(xT, yT, label="Tierra")
ax.plot(xM, yM, label="Marte")
ax.plot(0, 0, "o", color="orange", ms=12, label="Sol")
# una línea Tierra-Marte cada 30 días durante los primeros 5 meses
for d in range(0, 150, 30):
    ax.plot([xT[d], xM[d]], [yT[d], yM[d]], color="gray", lw=0.7)
ax.set_aspect("equal"); ax.legend(); ax.set_xlabel("x (ua)"); ax.set_ylabel("y (ua)")
ax.set_title("Órbitas vistas desde encima del Sistema Solar")
plt.show()

## 3. Marte visto desde la Tierra

Desde la Tierra no vemos las órbitas: vemos la **dirección** en la que está Marte. Es el ángulo del vector que va de la Tierra a Marte, $(x_M - x_T,\ y_M - y_T)$. La función `np.arctan2(y, x)` da ese ángulo teniendo en cuenta el cuadrante (la arcotangente normal se equivoca en la mitad de los casos).

A ese ángulo se le llama **longitud geocéntrica**: dice en qué punto del fondo de estrellas, a lo largo de la eclíptica, vemos el planeta. Cuando crece, Marte avanza hacia el este (movimiento directo). Cuando decrece, retrocede.

`np.unwrap` evita los saltos de 360° a 0° cuando el ángulo da la vuelta.

In [ ]:
dx, dy = xM - xT, yM - yT
longitud = np.degrees(np.unwrap(np.arctan2(dy, dx)))

plt.figure(figsize=(9, 4))
plt.plot(dias, longitud)
plt.xlabel("días desde la oposición inicial")
plt.ylabel("longitud geocéntrica de Marte (°)")
plt.title("La dirección de Marte vista desde la Tierra")
plt.grid(alpha=0.3)
plt.show()

La curva sube casi siempre, pero tiene tramos en los que **baja**: ahí está el movimiento retrógrado. Vamos a localizarlos. La velocidad angular aparente es la variación de la longitud de un día al siguiente (`np.diff`). Es negativa durante el retroceso.

In [ ]:
vel = np.diff(longitud)                 # grados por día
retro = vel < 0                         # True los días de retroceso

# Inicios y finales de cada tramo retrógrado
cambios = np.diff(retro.astype(int))
inicios = list(np.where(cambios == 1)[0] + 1)
finales = list(np.where(cambios == -1)[0] + 1)
if retro[0]:
    inicios = [0] + inicios             # la simulación empieza en plena oposición

print("Tramos retrógrados (días):")
for i, f in zip(inicios, finales):
    print(f"  del día {i:4d} al {f:4d}  → dura {f - i} días")

Como empezamos en plena oposición, el primer tramo está cortado por la mitad. El segundo está completo: esa es la duración de un bucle en este modelo. Las fechas reales varían un poco de una oposición a otra porque la órbita de Marte es una elipse bastante alargada.

**Ejercicio.** Calcula el tiempo entre los **centros** de dos tramos retrógrados consecutivos y compáralo con el periodo sinódico $S$ de la fórmula $1/S = 1/T_T - 1/T_M$.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
S = 1 / (1 / T_T - 1 / T_M)
print(f"Periodo sinódico con la fórmula: {S:.1f} días")
# El primer tramo empieza en la oposición (día 0); el segundo tiene su centro en la siguiente
centro_2 = (inicios[1] + finales[1]) / 2
print(f"Centro del segundo bucle en la simulación: día {centro_2:.1f}")

## 4. El bucle sobre el fondo de estrellas

En el cielo, el bucle no es solo un ir y volver: como las órbitas no están exactamente en el mismo plano, Marte sube o baja un poco y dibuja una lazada. Nuestro modelo es plano, así que solo vemos el ir y volver. Para ver la parte del movimiento que se repite, dibujamos la longitud en los 200 días alrededor de la segunda oposición.

In [ ]:
d0 = int(round(S))
ventana = slice(d0 - 100, d0 + 100)
plt.figure(figsize=(9, 3))
plt.plot(longitud[ventana] - longitud[d0], dias[ventana] - d0)
plt.axhline(0, color="gray", lw=0.5)
plt.gca().invert_xaxis()                 # en el cielo, el este queda a la izquierda mirando al sur
plt.xlabel("longitud relativa a la de la oposición (°)  ← este | oeste →")
plt.ylabel("días desde la oposición")
plt.title("Marte alrededor de la oposición: avanza, se para, retrocede y vuelve a avanzar")
plt.grid(alpha=0.3)
plt.show()

## 5. Experimenta

1. Cambia Marte por Júpiter ($r = 5{,}203$ ua, $T = 4332{,}6$ días). ¿Dura más o menos el retroceso? ¿Cada cuánto se repite?
2. Prueba con un planeta imaginario a 1,1 ua. ¿Qué pasa con el periodo sinódico cuando dos planetas tienen periodos parecidos?
3. (Más difícil) Para Venus, que va por dentro, cambia el papel de los dos planetas y busca cuándo retrocede. Comprueba que ocurre cerca de la **conjunción inferior**, cuando Venus pasa entre la Tierra y el Sol.

In [ ]:
# Tu experimento aquí